In [ ]:
import torch
import math


class LinearLayer:
    def __init__(self,in_features,out_features,bias=False):
        kaim = math.sqrt(2/in_features)
        self.weights = torch.randn(out_features,in_features)*kaim
        self.has_bias= bias
        if bias:
            self.bias = torch.zeros(out_features)
        else:
            self.bias = None

    def forward(self,x):
        self.x  = x
        out = self.x @ self.weights.T
        if self.has_bias:
            out = out+self.bias
        
        return out


    def backward(self,grad_out):
        x_shape = self.x.shape
        x_flat = self.x.flatten(0,-2)
        grad_flat = grad_out.flatten(0,-2)

        grad_inputs = grad_flat @ self.weights
        self.weights.grad = grad_flat.T @ x_flat
        if self.has_bias:
            self.bias.grad = grad_flat.sum(dim=0)

        grad_inputs = grad_inputs.reshape(x_shape)
        return grad_inputs


class LayerNorm:

    def __init__(self,num_dims,eps):
        self.eps = eps
        self.gamma = torch.ones(num_dims)
        self.beta = torch.zeros(num_dims)


    def forward(self,x):
        self.avg = torch.mean(x,dim=-1,keepdim=True)
        self.var = torch.mean( (x - self.avg)**2,dim=-1,keepdim=True )

        self.x_norm = (x - self.avg)/torch.sqrt(self.var+self.eps)
        out = self.x_norm * self.gamma + self.beta
        return out

    
    def backward(self, grad_out):
        self.grad_beta = torch.sum(grad_out, dim=(0, 1))
        self.grad_gamma = torch.sum(grad_out * self.x_norm, dim=(0, 1))
    
        grad_x = grad_out * self.gamma

        return (1.0 / torch.sqrt(self.var + self.eps)) * (
            grad_x
            - torch.mean(grad_x, dim=-1, keepdim=True)
            - self.x_norm * torch.mean(grad_x * self.x_norm, dim=-1, keepdim=True)
        )
        
        

class Relu:
    def forward(self,x):
        self.x = x
        return torch.clamp(x,min=0)

    def backward(self,grad_out):
        return grad_out * (self.x > 0)



class Dropout:
    
    def __init__(self,p=0.1,training=True):
        self.p = p
        self.training = training
        self.mask = None
        

    def forward(self,x):
        if self.training:
            self.mask = ((torch.rand_like(x) > self.p).float())/(1.0-self.p)
            return x * self.mask
        else:
            self.mask = None
            return x   

    def backward(self,grad_out):
        if self.training:
            return grad_out * self.mask
        else:
            return grad_out


        
class MyMlp:
    def __init__(self, in_features, hidden_features, out_features):
        self.linear1 = LinearLayer(in_features, hidden_features)
        self.relu = Relu()
        self.linear2 = LinearLayer(hidden_features, out_features)
        self.dropout = Dropout(p=0.1, training=True)

    def forward(self, x):
        x = self.linear1.forward(x)
        x = self.relu.forward(x)
        x = self.linear2.forward(x)
        x = self.dropout.forward(x)
        return x

    def backward(self, grad_out):
        grad_drop = self.dropout.backward(grad_out)
        grad_linear2 = self.linear2.backward(grad_drop)
        grad_relu = self.relu.backward(grad_linear2)
        grad_linear1 = self.linear1.backward(grad_relu)
        return grad_linear1


class softmax:

    def forward(self,scores):
        max_score = torch.max(scores,dim=-1,keepdim=True).values
        scores_exp = torch.exp(scores-max_score)
        self.scores_sum = scores_exp.sum(dim=-1,keepdim=True)
        self.out = scores_exp/self.scores_sum
        return self.out

    
    def backward(self,grad_attn):

        sum_term = (grad_attn *self.out ).sum(dim=-1, keepdim=True)
        grad_scores = self.out * (grad_attn - sum_term)

        return grad_scores
         

class CausalSelfAttention:
    def __init__(self,num_dims,num_heads):
        self.num_dims = num_dims
        self.num_heads = num_heads
        self.head_dims = num_dims // num_heads

        self.w_q = LinearLayer(num_dims,num_dims,bias=False)
        self.w_k = LinearLayer(num_dims,num_dims,bias=False)
        self.w_v = LinearLayer(num_dims,num_dims,bias=False)

        self.proj_out = LinearLayer(num_dims,num_dims,bias=True)
        self.softmax = softmax()

        self.attn_drop = Dropout(p=0.1,training=True)
        self.resid_drop = Dropout(p=0.1,training=True)

    def forward(self,x):
        B,T,D = x.shape

        Q = self.w_q.forward(x)
        K = self.w_k.forward(x)
        V = self.w_v.forward(x)

        Q = Q.view(B,T,self.num_heads,self.head_dims).transpose(1,2)
        K = K.view(B,T,self.num_heads,self.head_dims).transpose(1,2)
        V = V.view(B,T,self.num_heads,self.head_dims).transpose(1,2)

        self.Q = Q
        self.K = K
        self.V = V
        
        scores = Q @ K.transpose(-2,-1) / math.sqrt(self.head_dims)
        masks = torch.triu(torch.ones(T,T,dtype=torch.bool,device=scores.device),diagonal=1)

        scores = scores.masked_fill(masks,float("-inf"))

        self.attn_scores = self.softmax.forward(scores)
        self.attn_scores = self.attn_drop.forward(self.attn_scores)
        self.out = self.attn_scores @ self.V
        
        self.out = self.out.transpose(1,2).contiguous().view(B,T,D)
        self.out = self.proj_out(self.out)
        self.out = self.resid_drop.forward(self.out)
        return self.out

    def backward(self,grad_out):
        B,T,D = grad_out.shape
        grad_resid = self.resid_drop.backward(grad_out)
        grad_proj = self.proj_out.backward(grad_resid)
        grad_proj = grad_proj.view(B,T,self.num_heads,self.head_dims).transpose(1,2)
        
        grad_attn = grad_proj @ self.V.transpose(-2,-1)
        
        grad_v = self.attn_scores.transpose(-2,-1) @ grad_proj
        
        grad_probs = self.attn_drop.backward(grad_attn)
        grad_scores = self.softmax.backward(grad_probs)   

        scale = 1.0 / math.sqrt(self.head_dims)
        grad_scores = grad_scores * scale
        
        grad_q = grad_scores @ self.K
        grad_k = grad_scores.transpose(-2,-1) @ self.Q
        

        grad_Q = grad_q.transpose(1, 2).contiguous().view(B, T, D)
        grad_K = grad_k.transpose(1, 2).contiguous().view(B, T, D)
        grad_V = grad_v.transpose(1, 2).contiguous().view(B, T, D)

        grad_x_q = self.w_q.backward(grad_Q)
        grad_x_k = self.w_k.backward(grad_K)
        grad_x_v = self.w_v.backward(grad_V)

        grad_x = grad_x_q+grad_x_k+grad_x_v

        return grad_x

    def parameters(self):
        return [self.w_q.weights, self.w_k.weights, self.w_v.weights, self.proj_out.weights, self.proj_out.bias]
        

        
        